# Bài 16 · Cơ chế chú ý (Attention): đọc số tiếng Việt bằng cách nhìn lại đúng chỗ

[![Mở trong Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/trungnguyenvn/hoc-sau-notebooks/blob/main/16-attention.ipynb)

Notebook đi kèm [Bài 16 của Học Sâu · Reels](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/attention/) (reel và demo tương tác nằm ở trang bài học).
Bài toán là một phép “dịch” nhỏ mà máy kiểm tra được tuyệt đối: đọc cách đọc tiếng Việt của một số
(“hai nghìn không trăm hai mươi sáu”) rồi viết lại các chữ số (2026). Ta chạy bằng numpy chính các mô hình đã huấn luyện
mà demo dùng: một bộ mã hóa–giải mã GRU **không** attention, và cùng mô hình đó **có** attention của Bahdanau.

**Bạn sẽ làm:**
1. Viết bộ đọc số tiếng Việt và sinh tập kiểm tra, rồi chạy mô hình không attention: nó đọc sai ở đâu khi câu dài ra.
2. Tự tay tính một bước attention: điểm, softmax, trung bình có trọng số; vẽ bản đồ chú ý giữa hai câu.
3. Can thiệp vào trọng số chú ý để xem chúng có thật sự “giải thích” dự đoán không.
4. Đo vì sao Transformer chia $q \cdot k$ cho $\sqrt{d_k}$, trên vectơ ngẫu nhiên và trên trạng thái GRU thật.
5. Thay bộ mã hóa bằng một lớp tự chú ý (self-attention) và thấy nó không biết thứ tự từ nếu thiếu vectơ vị trí.

**Cần biết trước:** [Bài 15 · RNN & LSTM](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/rnn/) (GRU và các cổng) và
[Bài 04 · Softmax & hàm mất mát](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/ham-mat-mat/).
**Thời gian chạy:** dưới 1 phút trên CPU, kể cả tải hai tệp kết quả (~0,6 MB) lần đầu.

Mọi con số trong notebook đều do chính các ô bên dưới tính ra (hoặc đọc từ tệp kết quả mà demo dùng), và trùng với
số trong reel và demo.

> Các ô **🤔 Dự đoán trước** đặt một câu hỏi trước khi chạy. Hãy tự trả lời rồi mới chạy ô kế tiếp.

## 1 · Chuẩn bị

Chỉ cần numpy và matplotlib (Colab đã cài sẵn). Hàm `vn()` in số theo kiểu Việt Nam: dấu phẩy thập phân,
dấu chấm hàng nghìn. Python vẫn *gõ* số thập phân bằng dấu chấm (`0.1`), chỉ phần in ra là đổi.

In [ ]:
import base64
import hashlib
import itertools
import json
import math
import urllib.request
from collections import defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.ticker import FixedLocator, FuncFormatter, NullFormatter, NullLocator, ScalarFormatter

# Bảng màu của khóa học (an toàn cho người mù màu); luôn đi kèm một mã hóa thứ hai (hình dạng, nét gạch, nhãn)
BLUE, ORANGE, GREEN, GREY = "#3987e5", "#d95926", "#199e70", "#8a94a6"
WEIGHTS = LinearSegmentedColormap.from_list("trang_xanh", ["#ffffff", BLUE])  # trọng số chú ý: 0 → 1
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})


def vn(x, digits=2):
    """Số kiểu Việt Nam: 13.007 · 0,1 · −4,38 (không in “−0”)."""
    s = f"{round(float(x), digits) + 0.0:,.{digits}f}".replace(",", "_").replace(".", ",").replace("_", ".")
    return s.replace("-", "−")


def pct(p, digits=2):
    return vn(100 * p, digits) + "%"


def vn_tick(v, _pos=None):
    """Nhãn vạch trên trục: 0,5 · −2 · 60.000 (bỏ số 0 thừa ở cuối)."""
    if abs(v) < 1e-12:
        return "0"
    s = f"{v:.4f}".rstrip("0").rstrip(".")
    return vn(v, len(s.split(".")[1]) if "." in s else 0)


SUP = str.maketrans("-0123456789", "⁻⁰¹²³⁴⁵⁶⁷⁸⁹")


def log_tick(v, _pos=None):
    """Nhãn vạch trục log: 0,5 · 20 · 10⁻¹²."""
    if 0.01 <= v <= 1000:
        return vn_tick(v)
    e = math.floor(math.log10(v) + 1e-9)
    m = v / 10.0**e
    return (f"{vn_tick(m)}·" if abs(m - 1) > 1e-9 else "") + f"10{str(e).translate(SUP)}"


def vn_axes(fig):
    """Đổi nhãn mọi trục của hình (cả thanh màu) sang kiểu Việt Nam; gọi ngay trước plt.show().
    Trục log lấy vạch 1–2–5 nếu ngắn, hoặc các lũy thừa của 10 (cách đều, tối đa ~9 vạch) nếu dài."""
    for ax in fig.axes:
        for axis, lim in ((ax.xaxis, ax.get_xlim()), (ax.yaxis, ax.get_ylim())):
            if axis.get_scale() != "log":
                # only axes still on matplotlib's default number formatter: text labels and custom formatters stay
                if isinstance(axis.get_major_formatter(), ScalarFormatter):
                    axis.set_major_formatter(FuncFormatter(vn_tick))
                continue
            lo, hi = sorted(lim)
            k0, k1 = math.ceil(math.log10(lo) - 1e-9), math.floor(math.log10(hi) + 1e-9)
            if hi / lo < 300:
                ticks = [m * 10.0**k for k in range(k0 - 1, k1 + 1) for m in (1, 2, 5) if lo <= m * 10.0**k <= hi]
            else:
                step = max(1, math.ceil((k1 - k0) / 8))
                ticks = [10.0**k for k in range(k0, k1 + 1) if k % step == 0]
                if step > 1:
                    axis.set_minor_locator(NullLocator())
            axis.set_major_locator(FixedLocator(ticks))
            axis.set_minor_formatter(NullFormatter())
            axis.set_major_formatter(FuncFormatter(log_tick))


print(vn(53259, 0), vn(0.7129, 2), pct(0.7233))

## 2 · Dữ liệu: đọc một số bằng tiếng Việt

Câu nguồn là cách đọc chuẩn miền Bắc của một số nguyên có 1–12 chữ số; câu đích là các chữ số của nó.
Luật đọc có đủ các chỗ khó của tiếng Việt:
- **mươi / mười:** 20 là “hai mươi”, 10 là “mười”;
- **mốt, tư, lăm:** 21 là “hai mươi mốt”, 24 là “hai mươi tư”, 15 là “mười lăm”, 25 là “hai mươi lăm”;
- **linh:** 105 là “một trăm linh năm”;
- **không trăm:** nhóm ba chữ số ở giữa luôn đọc đủ hàng trăm (2026 là “hai nghìn **không trăm** hai mươi sáu”);
- **bỏ nhóm 000:** 2.000.000 chỉ là “hai triệu”.

Toàn bộ từ vựng chỉ có 20 từ. Ô dưới viết bộ đọc (giống hệt `reel/attention/numvi.py` của khóa học) và kiểm tra nó
trên vài số.

In [ ]:
D = ["không", "một", "hai", "ba", "bốn", "năm", "sáu", "bảy", "tám", "chín"]
SCALE = ["", "nghìn", "triệu", "tỷ"]


def tens_units(b, c):
    """Hàng chục b và hàng đơn vị c của một nhóm ba chữ số."""
    if b == 0:
        return ["linh", D[c]] if c else []
    if b == 1:
        return ["mười"] + (["lăm" if c == 5 else D[c]] if c else [])
    return [D[b], "mươi"] + ([{1: "mốt", 4: "tư", 5: "lăm"}.get(c, D[c])] if c else [])


def group(g, lead):
    a, b, c = g // 100, g // 10 % 10, g % 10
    if lead and a == 0:  # nhóm đầu tiên không đọc số 0 ở đầu: 26 là "hai mươi sáu", không phải "không trăm…"
        return [D[c]] if b == 0 else tens_units(b, c)
    return [D[a], "trăm"] + tens_units(b, c)


def read(n):
    """Số nguyên n ≥ 0 → danh sách các từ."""
    if n == 0:
        return ["không"]
    groups = []
    while n:
        groups.append(n % 1000)
        n //= 1000
    words = []
    for k in reversed(range(len(groups))):
        lead = k == len(groups) - 1
        if groups[k] == 0 and not lead:  # bỏ nhóm 000 ở giữa
            continue
        words += group(groups[k], lead)
        if SCALE[k]:
            words.append(SCALE[k])
    return words

Kiểm tra bộ đọc trên các số có đủ mọi luật khó (mốt, tư, lăm, linh, không trăm, bỏ nhóm 000), rồi đếm từ vựng.

In [ ]:
CHECKS = {
    15: "mười lăm",
    21: "hai mươi mốt",
    24: "hai mươi tư",
    105: "một trăm linh năm",
    2026: "hai nghìn không trăm hai mươi sáu",
    2000000: "hai triệu",
    1000005: "một triệu không trăm linh năm",
    1001000: "một triệu không trăm linh một nghìn",
}
for n, text in CHECKS.items():
    print(f"{vn(n, 0):>10}  {' '.join(read(n))}")
    assert " ".join(read(n)) == text, "khác với numvi.py của reel và demo"

vocab = {w for n in range(1000) for w in read(n)} | {"nghìn", "triệu", "tỷ"}  # 0–999 đã dùng đủ mọi từ trong nhóm
print(f"{len(vocab)} từ:", " · ".join(sorted(vocab)))
assert len(vocab) == 20

Mỗi số lấy ngẫu nhiên có đúng $d$ chữ số. Nếu chữ số đều ngẫu nhiên, một nhóm 000 chỉ có xác suất $0{,}001$ và mô hình
không bao giờ học đọc số tròn. Nên bộ lấy mẫu của khóa học cho mỗi chữ số bằng 0 với xác suất 0,25, và mỗi nhóm ba
chữ số bên dưới nhóm đầu bằng 000 với xác suất 0,2.

Tập kiểm tra là 300 số cho mỗi độ dài $d = 1 \dots 12$, rút từ `default_rng(2000 + d)`, đúng tập kiểm tra của
`reel/attention/train.py` (tập kiểm định dùng `default_rng(1000 + d)` để chọn $\eta$; mô hình học trên các số rút mới ở mỗi bước).

In [ ]:
P_ZERO, P_GROUP, N_TEST, D_MAX = 0.25, 0.2, 300, 12
SRC = ["<pad>", *D, "mốt", "tư", "lăm", "mười", "mươi", "trăm", "linh", "nghìn", "triệu", "tỷ"]  # id 0 là phần đệm
W2I = {w: i for i, w in enumerate(SRC)}
EOS, BOS = 10, 11  # đầu ra: 0–9 là chữ số, 10 là ⟨hết⟩; bộ giải mã bắt đầu từ ký hiệu 11


def sample(rng, d, n, p0=P_ZERO, pg=P_GROUP):
    """n số có đúng d chữ số, mỗi hàng là các chữ số."""
    digs = np.where(rng.random((n, d)) < p0, 0, rng.integers(1, 10, (n, d)))
    digs[:, 0] = rng.integers(1, 10, n)  # chữ số đầu khác 0
    for g0 in range(d - 3, 0, -3):  # các nhóm ba chữ số tính từ bên phải, trừ nhóm đầu
        digs[rng.random(n) < pg, g0 : g0 + 3] = 0
    return digs


def encode_set(digs):
    """Các hàng chữ số → id từ nguồn X (đệm 0), mặt nạ M (1 = có từ), đích Y (các chữ số rồi ⟨hết⟩)."""
    srcs = [[W2I[w] for w in read(int("".join(map(str, r))))] for r in digs]
    X = np.zeros((len(srcs), max(map(len, srcs))), int)
    for i, s in enumerate(srcs):
        X[i, : len(s)] = s
    return X, (X > 0).astype(float), np.concatenate([digs, np.full((len(srcs), 1), EOS)], 1)


def as_int(row):
    return int("".join(map(str, row)))


TEST = {d: encode_set(sample(np.random.default_rng(2000 + d), d, N_TEST)) for d in range(1, D_MAX + 1)}
for d in (4, 7, 12):
    n = as_int(TEST[d][2][0, :-1])
    print(f"d = {d:>2}: {vn(n, 0):>17} → {' '.join(read(n))}")
X12, M12, Y12 = TEST[12]
has000 = [(TEST[d][2][:, 1:-1].reshape(-1) == 0).mean() for d in range(2, D_MAX + 1)]
print(f"tỉ lệ chữ số 0 (trừ chữ số đầu): {pct(np.mean(has000), 1)}")
NB = as_int(Y12[90, :-1])  # ví dụ của reel: hàng 90 của tập 12 chữ số
print(f"số thứ 90 của tập 12 chữ số: {vn(NB, 0)}, đọc bằng {len(read(NB))} từ")
assert NB == 964131195884 and len(read(NB)) == 23, "khác với reel: hãy kiểm tra lại bộ lấy mẫu"

Câu nguồn dài ra theo số chữ số: một số 12 chữ số cần tới 23 từ. Hình dưới vẽ số từ trung bình (đường xanh, chấm tròn)
và khoảng ngắn nhất–dài nhất (dải xám) của 300 câu kiểm tra ở mỗi độ dài.

In [ ]:
lens = {d: TEST[d][1].sum(1) for d in TEST}
ds = np.arange(1, D_MAX + 1)
fig, ax = plt.subplots(figsize=(7, 3))
ax.fill_between(
    ds, [lens[d].min() for d in ds], [lens[d].max() for d in ds], color=GREY, alpha=0.25, label="ngắn nhất–dài nhất"
)
ax.plot(ds, [lens[d].mean() for d in ds], "o-", color=BLUE, label="trung bình")
ax.set(xlabel="số chữ số d", ylabel="số từ trong câu nguồn", xticks=ds, title="Câu nguồn dài ra theo số chữ số")
ax.legend(frameon=False)
ax.grid(axis="y", alpha=0.3)
vn_axes(fig)
plt.show()
print("dài nhất:", int(max(lens[d].max() for d in ds)), "từ")

Các mô hình đã được huấn luyện sẵn (6.000 bước Adam, mỗi bước 64 số mới; quá lâu cho notebook).
Ô dưới tải đúng hai tệp mà demo dùng và kiểm tra MD5:
- `attention-weights.json`: trọng số float16 của seed 0, H = 64, cho bốn mô hình: không attention, attention cộng
  tính, tự chú ý không vị trí và có vị trí;
- `attention-s2s.json`: kết quả kiểm tra của cả lần quét (10 cấu hình × 3 giá trị $\eta$ × 3 seed), dùng ở mục 3, 7 và 8.

In [ ]:
# Kết quả của một lần huấn luyện dài (cùng tệp với demo trên web), tải từ thư mục data/ của repo notebook công khai.
def fetch_json(name, md5):
    p = Path("data") / name
    if not p.exists():
        p.parent.mkdir(exist_ok=True)
        urllib.request.urlretrieve("https://raw.githubusercontent.com/trungnguyenvn/hoc-sau-notebooks/main/data/" + name, p)
    assert hashlib.md5(p.read_bytes()).hexdigest() == md5, f"{name} bị hỏng, hãy xóa và tải lại"
    return json.loads(p.read_text())


shipped = fetch_json("attention-weights.json", "fd5de1ef245d385fbbabedc5c5f2c10d")  # trọng số của demo
sweep = fetch_json("attention-s2s.json", "1a66caffd83e34411148c3b2bb3c0d3a")  # kết quả quét của demo
size = sum((Path("data") / n).stat().st_size for n in ("attention-weights.json", "attention-s2s.json"))
print(
    "mô hình:",
    list(shipped["models"]),
    "· số lần chạy trong tệp quét:",
    len(sweep["runs"]),
    f"· {vn(size / 1e6, 1)} MB",
)

## 3 · Nhồi cả câu vào một vectơ

Bộ mã hóa là một GRU (Bài 15) đọc từng từ, mỗi từ cập nhật trạng thái ẩn $h_j$. Bộ giải mã là một GRU khác, viết từng
chữ số. Ở mô hình **không attention** (bộ mã hóa–giải mã RNN của Cho và cộng sự, 2014), bộ giải mã chỉ thấy trạng
thái cuối cùng $h_T$: nó là trạng thái ban đầu của bộ giải mã và cũng là “ngữ cảnh” $c$ ở mọi bước. Mọi điều về 23 từ
phải lọt qua đúng 64 số đó.

Ô dưới viết lại hai mạng bằng numpy theo quy ước hàng $z = xW + b$, giống `reel/attention/s2s.py`:
- `Ex[x]` là hàng của từ $x$ trong bảng embedding, đã nhân sẵn với trọng số đầu vào của cả ba phần GRU (vì $x$ là
  vectơ one-hot, $xW$ chính là một hàng của $W$);
- bộ giải mã nhận chữ số vừa viết `Ey[y]` cùng ngữ cảnh $c$, và tính logits $s_i W_o + c_i W_{co} + b_o$ cho 11 đầu ra
  (10 chữ số và ⟨hết⟩).

In [ ]:
def unpack(model):
    """float16 mã hóa base64 → mảng float64, như demo."""
    return {
        k: np.frombuffer(base64.b64decode(v["f16"]), "<f2").astype(np.float64).reshape(v["shape"])
        for k, v in model.items()
    }


MODELS = {name: unpack(m) for name, m in shipped["models"].items()}
assert shipped["src"] == SRC


def sig(v):
    return 0.5 * (1 + np.tanh(0.5 * v))  # = 1 / (1 + e^(−v)), không bao giờ tràn số


def gru(ax, h, U):
    """Một bước GRU. ax = xW + b của cả ba phần (cổng đặt lại r, cổng cập nhật z, giá trị đề cử n), nối theo cột."""
    H = h.shape[1]
    rz = sig(ax[:, : 2 * H] + h @ U[:, : 2 * H])
    r, z = rz[:, :H], rz[:, H:]
    n = np.tanh(ax[:, 2 * H :] + (r * h) @ U[:, 2 * H :])
    return z * h + (1 - z) * n  # z gần 1: giữ trạng thái cũ (quy ước của Cho và cộng sự, như Bài 15)


def encode(p, X, M):
    """Bộ mã hóa GRU: trạng thái sau mỗi từ Hs (B, S, H) và trạng thái cuối h_T. Phần đệm (M = 0) giữ nguyên h."""
    h = np.zeros((X.shape[0], p["Ue"].shape[0]))
    Hs = np.zeros((*X.shape, h.shape[1]))
    for j in range(X.shape[1]):
        m = M[:, j : j + 1]
        h = m * gru(p["Ex"][X[:, j]] + p["be"], h, p["Ue"]) + (1 - m) * h
        Hs[:, j] = h
    return Hs, h


for name, p in MODELS.items():
    print(f"{name:>5}: {vn(sum(v.size for v in p.values()), 0):>6} tham số")
assert [sum(v.size for v in p.values()) for p in MODELS.values()] == [45003, 53259, 50379, 51915]

Bộ giải mã viết **tham lam** (greedy): ở mỗi bước chọn đầu ra có logit lớn nhất, rồi đưa chữ số đó vào bước sau. Hàm
`decode` dưới đây phục vụ cả hai mô hình; phần attention (`att="add"`) sẽ được giải thích ở mục 4, giờ chỉ cần biết
với `att="none"` ngữ cảnh luôn là $c = h_T$.

Một số được tính là đọc đúng khi **mọi** chữ số và cả ký hiệu ⟨hết⟩ đều đúng (exact match, như demo).

In [ ]:
def softmax_masked(e, M):
    """Softmax theo hàng, bỏ qua phần đệm (M = 0)."""
    e = np.where(M > 0, e, -np.inf)
    a = np.exp(e - e.max(-1, keepdims=True))
    return a / a.sum(-1, keepdims=True)


def additive_scores(p, s, keys):
    """Điểm của Bahdanau: e_j = tanh(s W_a + h_j U_a) · v_a cho mọi vị trí j. keys = Hs @ U_a, tính sẵn."""
    return np.tanh(keys + (s @ p["Wa"])[:, None, :]) @ p["va"]


def decode(p, Hs, s, M, steps, att, edit=None):
    """Giải mã tham lam. Trả về các đầu ra (B, steps) và trọng số chú ý (B, steps, S) nếu có attention.
    edit(a, M) cho phép sửa trọng số chú ý trước khi dùng (mục 6)."""
    y, h_T = np.full(Hs.shape[0], BOS), s
    keys = Hs @ p["Ua"] if att == "add" else None  # h_j U_a không phụ thuộc bước i: tính một lần
    out, A = [], []
    for _ in range(steps):
        if att == "none":
            c = h_T  # nút thắt: cùng một vectơ ở mọi bước
        else:
            a = softmax_masked(additive_scores(p, s, keys), M)
            a = edit(a, M) if edit else a
            A.append(a)
            c = np.einsum("bs,bsh->bh", a, Hs)  # trung bình có trọng số của các h_j
        s = gru(p["Ey"][y] + c @ p["Wc"] + p["bd"], s, p["Ud"])
        y = (s @ p["Wo"] + c @ p["Wco"] + p["bo"]).argmax(1)
        out.append(y)
    return np.stack(out, 1), (np.stack(A, 1) if A else None)


def run(name, X, M, steps, edit=None):
    p = MODELS[name]
    Hs, h_T = encode(p, X, M)
    return decode(p, Hs, h_T, M, steps, "none" if name == "none" else "add", edit)


def digits_of(row):
    """Các đầu ra đến trước ⟨hết⟩, ghép thành chuỗi chữ số."""
    row = [int(v) for v in row]
    return "".join(map(str, row[: row.index(EOS)] if EOS in row else row))

Chạy mô hình không attention trên cả 3.600 số kiểm tra (300 cho mỗi độ dài). Ô cũng so số lần đọc đúng với kết quả
seed 0 mà lần quét đã lưu, để chắc rằng bản numpy này chính là mô hình của demo.

In [ ]:
def exact(name, d, edit=None):
    X, M, Y = TEST[d]
    return (run(name, X, M, Y.shape[1], edit)[0] == Y).all(1)


ok_none = {d: exact("none", d) for d in TEST}
print("không attention, số đọc đúng trên 300 ở mỗi độ dài:")
print("  ".join(f"d{d}: {ok_none[d].sum()}" for d in TEST))
print(f"12 chữ số: {pct(ok_none[12].mean())}")


def stored(att, H=64, steps=6000):
    """Một cấu hình trong tệp quét của demo."""
    return next(r for r in sweep["runs"] if r["att"] == att and r["H"] == H and r["steps"] == steps)


def span(vals, f=pct):
    """Khoảng thấp nhất–cao nhất giữa các seed."""
    lo, hi = min(vals), max(vals)
    return f(lo) if f(lo) == f(hi) else f"{f(lo)}–{f(hi)}"


def seed0_counts(att):
    return [round(N_TEST * v) for v in stored(att)["seeds"][0]["test"]["em"]]


assert [int(ok_none[d].sum()) for d in TEST] == seed0_counts("none"), "khác với demo: hãy kiểm tra lại mô hình"

Đến 8 chữ số, vectơ cố định gần như không sai; từ 9 chữ số trở đi số lỗi tăng nhanh, đến 12 chữ số (câu tới 23 từ)
chỉ còn 72,33% số được đọc đúng. Đây là đúng điều Cho và cộng sự (2014) thấy khi dịch máy: chất lượng giảm nhanh khi
câu dài ra.

### 🤔 Dự đoán trước
Mô hình không attention đọc sai 83 số 12 chữ số. Bạn nghĩ lỗi nằm chủ yếu ở **chữ số nào** (đầu, giữa hay cuối)?
Và lỗi thường là **kiểu gì**: viết sai một chữ số, thiếu chữ số, hay đảo chỗ hai chữ số?

In [ ]:
out12, _ = run("none", X12, M12, 13)
wrong = np.flatnonzero(~ok_none[12])
per_pos = (out12[wrong] != Y12[wrong]).sum(0)[:12]
swaps = 0
for b in wrong:
    diff = np.flatnonzero(out12[b] != Y12[b])
    if len(diff) == 2 and diff[1] == diff[0] + 1 and (out12[b, diff] == Y12[b, diff[::-1]]).all():
        swaps += 1
print(f"{len(wrong)} số sai; số lỗi ở chữ số thứ 1…12:", per_pos.tolist())
print(f"{swaps} trong {len(wrong)} số sai chỉ vì đảo chỗ hai chữ số liền nhau")
assert len(wrong) == 83

fig, ax = plt.subplots(figsize=(7, 2.8))
ax.bar(np.arange(1, 13), per_pos, color=ORANGE, hatch="//", edgecolor="white")
ax.set(xlabel="vị trí chữ số (1 = chữ số đầu tiên)", ylabel="số lỗi", xticks=np.arange(1, 13))
ax.set_title("Không attention, 12 chữ số: các chữ số bị viết sai nằm ở đâu")
vn_axes(fig)
plt.show()

Lỗi dồn vào **giữa** số (chữ số thứ 5–10), và phần lớn là đảo chỗ hai chữ số liền nhau: mô hình vẫn nhớ *có những chữ
số nào*, nhưng 64 số của $h_T$ không giữ nổi chúng theo đúng *thứ tự*. Hai chữ số đầu gần như không sai, các chữ số
cuối sai ít hơn ở giữa: đó là những phần dễ giữ nhất, phần mở đầu và phần GRU vừa đọc xong.

Ví dụ của reel là số kiểm tra 964.131.195.884 (23 từ):

In [ ]:
words_b = read(NB)
Xb = np.array([[W2I[w] for w in words_b]])
Mb = np.ones(Xb.shape)
read_none, read_add = (digits_of(run(name, Xb, Mb, 14)[0][0]) for name in ("none", "add"))
wrong_at = [k for k, (o, t) in enumerate(zip(read_none, str(NB))) if o != t]
print(" ".join(words_b))
print(f"không attention: {vn(int(read_none), 0)}   (sai ở chữ số thứ {[k + 1 for k in wrong_at]})")
print(f"có attention:    {vn(int(read_add), 0)}")
assert read_none == "964131915884" and wrong_at == [6, 7] and read_add == str(NB), "khác với reel"

“195” thành “915”: đúng kiểu đảo chỗ ở trên. Mô hình có attention đọc đúng số đó (mục 4–5 xem nó làm thế nào).

Nhưng vectơ cố định là một **bức tường** thật, hay chỉ làm việc học khó hơn? Lần quét của khóa học trả lời. Hình dưới
vẽ tỉ lệ đọc đúng theo số chữ số, ở H = 64, cho 3 seed (đường: trung vị, dải: thấp nhất–cao nhất). Cam, ô vuông: không
attention sau 6.000 bước; xanh lá, tam giác, nét đứt: cùng mô hình học lâu gấp 4 (24.000 bước); xanh dương, chấm tròn:
có attention sau 6.000 bước.

In [ ]:
def em_seeds(att, H=64, steps=6000):
    """Tỉ lệ đọc đúng ở d = 1…12 của 3 seed, mảng (3, 12)."""
    return np.array([s["test"]["em"] for s in stored(att, H, steps)["seeds"]])


def em12(att, H=64, steps=6000):
    return span(em_seeds(att, H, steps)[:, -1])


fig, ax = plt.subplots(figsize=(8, 3.6))
for att, steps, color, marker, ls, label in (
    ("add", 6000, BLUE, "o", "-", "có attention, 6.000 bước"),
    ("none", 24000, GREEN, "^", "--", "không attention, 24.000 bước"),
    ("none", 6000, ORANGE, "s", "-", "không attention, 6.000 bước"),
):
    em = 100 * em_seeds(att, steps=steps)
    ax.fill_between(ds, em.min(0), em.max(0), color=color, alpha=0.2)
    ax.plot(ds, np.median(em, 0), marker=marker, ls=ls, color=color, label=label)
ax.set(xlabel="số chữ số d", ylabel="đọc đúng cả số (%)", xticks=ds, ylim=(60, 101))
ax.set_title("H = 64, 3 seed: nút thắt chỉ lộ ra ở số dài")
ax.legend(frameon=False, loc="lower left")
ax.grid(axis="y", alpha=0.3)
vn_axes(fig)
plt.show()

print("đọc đúng ở 12 chữ số (3 seed) theo cỡ trạng thái H:")
print(
    f"{'H':>4} {'tham số không / có att.':>24} {'không att. 6.000 bước':>22} {'24.000 bước':>15} {'có att. 6.000':>15}"
)
for H in (32, 64, 128):
    n_par = f"{vn(stored('none', H)['params'], 0)} / {vn(stored('add', H)['params'], 0)}"
    print(f"{H:>4} {n_par:>24} {em12('none', H):>22} {em12('none', H, 24000):>15} {em12('add', H):>15}")

- Sau 6.000 bước, attention với trạng thái chỉ 32 chiều (15.371 tham số) đọc đúng 99,67–100,00% số 12 chữ số, hơn
  vectơ cố định 128 chiều (163.723 tham số, 94,33–96,67%).
- Nhưng cho học lâu gấp 4, vectơ cố định 64 chiều lên 98,33–98,67%, và 128 chiều lên 100,00%. Nút thắt làm việc học
  **chậm và khó hơn**, chứ không phải một bức tường: 64 số đủ chứa một số 12 chữ số, chỉ là khó học cách nhồi nó vào.
- Vectơ nhỏ hơn thì nút thắt chặt thật: 32 chiều, kể cả 24.000 bước, chỉ đạt 72,00–79,67%.

Điều này khớp với Sutskever và cộng sự (2014): LSTM lớn của họ, không có attention, “không gặp khó với câu dài”
(họ đảo ngược câu nguồn để việc học dễ hơn).

Attention làm bài toán *dễ học* hơn hẳn. Mục sau xem nó làm điều đó thế nào.

## 4 · Nhìn lại câu nguồn: một bước attention bằng tay

Bahdanau, Cho & Bengio (2014) cho bộ giải mã nhìn lại **mọi** trạng thái $h_1, \dots, h_S$ của bộ mã hóa ở mỗi bước
$i$, thay vì chỉ $h_T$. Ba bước, theo quy ước hàng:

$$e_{ij} = \tanh(s_{i-1} W_a + h_j U_a)\, v_a, \qquad \alpha_{ij} = \frac{\exp e_{ij}}{\sum_{j'} \exp e_{ij'}}, \qquad c_i = \sum_j \alpha_{ij}\, h_j$$

- **Truy vấn (Query)** là $s_{i-1}$, trạng thái bộ giải mã trước khi viết chữ số $i$: “tôi đang tìm gì”.
- **Khóa (Key)** của vị trí $j$ là $h_j U_a$; một mạng nhỏ một lớp ẩn chấm điểm $e_{ij}$ mức khớp giữa truy vấn và khóa.
- **Softmax** (Bài 04) biến các điểm thành **trọng số chú ý** $\alpha_{ij}$: dương, cộng lại bằng 1 theo $j$.
- **Giá trị (Value)** là chính $h_j$; ngữ cảnh $c_i$ là trung bình có trọng số của chúng. Ở Bahdanau, khóa và giá trị
  đến từ cùng một vectơ $h_j$.

Mọi phép tính đều khả vi, nên các ma trận $W_a, U_a, v_a$ học được bằng gradient cùng phần còn lại của mạng.
Ô dưới tính bước đầu tiên khi đọc “hai nghìn không trăm hai mươi sáu”, bằng một vòng lặp qua 7 từ.

In [ ]:
p = MODELS["add"]
words_a = read(2026)
Xa = np.array([[W2I[w] for w in words_a]])
Ma = np.ones(Xa.shape)
Hs_a, s0 = encode(p, Xa, Ma)  # 7 trạng thái h_j (mỗi cái 64 số); bộ giải mã bắt đầu từ s_0 = h_T

query = s0[0] @ p["Wa"]  # phần của truy vấn, dùng chung cho mọi j
e = np.zeros(len(words_a))
for j in range(len(words_a)):
    key = Hs_a[0, j] @ p["Ua"]  # khóa của vị trí j
    e[j] = np.tanh(query + key) @ p["va"]  # điểm khớp e_1j
alpha = np.exp(e - e.max())  # trừ max không đổi kết quả, chỉ tránh tràn số
alpha /= alpha.sum()
context = alpha @ Hs_a[0]  # giá trị = h_j

for w, ej, aj in zip(words_a, e, alpha):
    print(f"{w:>6}: e = {vn(ej, 2):>6}   α = {pct(aj):>6}  {'█' * round(40 * aj)}")
print(f"tổng các α = {vn(alpha.sum(), 4)}; ngữ cảnh c₁ có {context.size} số")
vec = softmax_masked(additive_scores(p, s0, Hs_a @ p["Ua"]), Ma)[0]
assert np.allclose(alpha, vec)  # vòng lặp = bản vectơ hóa trong decode()
assert abs(e[0] - 13.0327) < 1e-3 and np.allclose(alpha[:2], [0.7129, 0.2744], atol=5e-4), "khác với reel"

Để viết chữ số đầu tiên, bộ giải mã đặt 71,29% trọng số vào “hai” và 27,44% vào “nghìn”: đúng hai từ quyết định
chữ số đầu là 2 và số có bốn chữ số. Điểm $e$ chênh nhau vài đơn vị là đủ: softmax lấy mũ, nên điểm hơn 1 đơn vị
nghĩa là trọng số gấp $e \approx 2{,}7$ lần.

## 5 · Bản đồ chú ý giữa hai câu

Lặp lại bước đó cho mọi chữ số, ta được một **bản đồ chú ý**: mỗi hàng là một bước viết, mỗi cột là một từ nguồn,
ô càng xanh thì trọng số càng lớn; số trong ô là trọng số (%). Mỗi hàng cộng lại bằng 1.

In [ ]:
def show_map(words, out, A, title, min_label=0.005):
    """Bản đồ chú ý: hàng = bước viết (chữ số đã viết), cột = từ nguồn; ghi % vào các ô ≥ min_label."""
    labels = [("⟨hết⟩" if v == EOS else str(v)) for v in out]
    _, ax = plt.subplots(figsize=(min(10, max(4, 0.42 * len(words) + 1.5)), 0.36 * len(labels) + 1.6))
    ax.imshow(A, cmap=WEIGHTS, vmin=0, vmax=1, aspect="auto")
    ax.set_xticks(range(len(words)), labels=words, rotation=60, ha="right")
    ax.set_yticks(range(len(labels)), labels=labels)
    ax.set(xlabel="từ nguồn", ylabel="chữ số viết ra", title=title)
    for i, row in enumerate(A):
        for j, v in enumerate(row):
            if v >= min_label:
                ax.text(
                    j, i, f"{100 * v:.0f}", ha="center", va="center", fontsize=8, color="white" if v > 0.6 else "black"
                )
    return ax


out_a, A_a = run("add", Xa, Ma, 14)
n_out = len(digits_of(out_a[0])) + 1  # các chữ số và ⟨hết⟩
show_map(words_a, out_a[0, :n_out], A_a[0, :n_out], "2026: trọng số chú ý (%)")
plt.show()
top = A_a[0, :n_out].argmax(1)
print("từ nặng nhất ở mỗi bước:", [words_a[j] for j in top], " · đọc ra:", digits_of(out_a[0]))
assert digits_of(out_a[0]) == "2026" and top.tolist() == [0, 2, 4, 6, 6], "khác với reel"
assert np.allclose(A_a[0].sum(1), 1)

Mỗi chữ số “nhìn” vào đúng chỗ của nó trong câu: 2 ở “hai”, 0 ở “không trăm”, 2 ở “hai (mươi)”, 6 ở “sáu”.
Không ai dạy mô hình gióng hàng từ với chữ số; bản đồ này tự xuất hiện vì nó giúp giảm hàm mất mát.
Bahdanau và cộng sự gọi đây là vừa học gióng hàng (align) vừa dịch.

### 🤔 Dự đoán trước
Với số dài 964.131.195.884, câu nguồn bắt đầu “chín trăm sáu mươi **tư** **tỷ** một trăm…”. Khi viết chữ số thứ ba
(số 4, đến từ “tư”), bộ giải mã đặt nhiều trọng số nhất vào từ nào?

In [ ]:
out_b, A_b = run("add", Xb, Mb, 14)
n_b = len(digits_of(out_b[0])) + 1
show_map(
    words_b, out_b[0, :n_b], A_b[0, :n_b], "964.131.195.884: trọng số chú ý (%), chỉ ghi các ô ≥ 5%", min_label=0.05
)
plt.show()
for i in (2, 6):  # bước 3 viết chữ số 4 (của “tư”), bước 7 viết chữ số 1 (đầu nhóm “một trăm chín mươi lăm”)
    a_i = A_b[0, i]
    top3 = " · ".join(f"“{words_b[j]}” (từ {j + 1}) {pct(a_i[j])}" for j in np.argsort(-a_i)[:3])
    print(f"bước {i + 1} (viết {out_b[0, i]}): {top3}")
print("từ nặng nhất ở mỗi bước:", " · ".join(words_b[j] for j in A_b[0, :n_b].argmax(1)))
assert A_b[0, :n_b].argmax(1).tolist() == [0, 2, 5, 6, 8, 10, 11, 14, 16, 18, 20, 22, 22], "khác với reel"

Không phải “tư”, mà là “tỷ”, từ *ngay sau* nó. Lý do: $h_j$ không phải là từ thứ $j$, mà là trạng thái GRU **sau khi đã
đọc** các từ $1 \dots j$. Trạng thái ở “tỷ” vừa đọc xong “…sáu mươi tư tỷ”, nên nó mang cả chữ số 4 lẫn vị trí của
nhóm (hàng tỷ). Mô hình học cách nhìn vào những trạng thái tiện nhất *cho nó*, không nhất thiết là từ mà người đọc sẽ
chỉ vào. Ở bước 7 (chữ số 1 mở đầu “một trăm chín mươi lăm nghìn”), trọng số chia gần đều giữa “triệu”, từ đứng
trước, và “một”.

(Bahdanau và cộng sự dùng RNN hai chiều, nên $h_j$ của họ tóm tắt cả các từ trước lẫn sau $j$. Bộ mã hóa ở đây chỉ
đọc từ trái sang phải, nên trạng thái ở một từ chỉ biết các từ đến đó.)

Vậy bản đồ chú ý cho biết mô hình **nhìn vào đâu**, nhưng đọc nó như “chữ số này đến từ từ nào” thì dễ sai. Mục sau
kiểm tra điều đó bằng can thiệp trực tiếp.

## 6 · Trọng số chú ý có giải thích được dự đoán không?

Jain & Wallace (2019) cho rằng attention **không** phải là lời giải thích (trên các bài phân loại, hỏi đáp và suy diễn):
trọng số thường lệch với các thước đo độ quan trọng khác, và những bản đồ rất khác nhau vẫn cho cùng dự đoán.
Wiegreffe & Pinter (2019) phản biện rằng điều đó tùy định nghĩa “giải thích”, và đề xuất các phép thử chặt hơn. Bài toán
đọc số cho ta làm thử ngay vài phép can thiệp mà không cần tranh luận: sửa đầu vào hoặc sửa trọng số, rồi xem đầu ra.

In [ ]:
w_ba = list(words_b)
w_ba[4] = "ba"  # "…sáu mươi tư tỷ…" → "…sáu mươi ba tỷ…"
X_ba = np.array([[W2I[w] for w in w_ba]])
print("đổi “tư” → “ba”:  ", vn(int(digits_of(run("add", X_ba, Mb, 14)[0][0])), 0))


def drop_word(j):
    """Đặt trọng số của vị trí j về 0 ở mọi bước, rồi chuẩn hóa lại các trọng số còn lại."""

    def edit(a, M):
        a = a.copy()
        a[:, j] = 0
        return a / a.sum(1, keepdims=True)

    return edit


for j in (4, 5):
    print(f"cấm nhìn vào “{words_b[j]}”:", vn(int(digits_of(run("add", Xb, Mb, 14, edit=drop_word(j))[0][0])), 0))
assert digits_of(run("add", X_ba, Mb, 14)[0][0]) == "963131195884"
assert digits_of(run("add", Xb, Mb, 14, edit=drop_word(4))[0][0]) == str(NB)

Đổi “tư” thành “ba” thì chữ số thứ ba thành 3: dự đoán **phụ thuộc** vào “tư”, dù “tư” chỉ nhận 10,71% trọng số.
Còn cấm hẳn bộ giải mã nhìn vào “tư” thì nó vẫn đọc đúng cả số, vì thông tin của “tư” đã nằm sẵn trong trạng thái ở
“tỷ”. Trọng số thấp không có nghĩa là không quan trọng.

### 🤔 Dự đoán trước
Thay trọng số chú ý bằng trung bình **đều** (mọi từ như nhau) ở mọi bước. Trên 300 số 12 chữ số, mô hình còn đọc đúng
bao nhiêu phần trăm? Và nếu chỉ giữ **một** từ nặng nhất ở mỗi bước (attention “cứng”)?

In [ ]:
def uniform(a, M):
    return M / M.sum(1, keepdims=True)


def top_k(k):
    """Chỉ giữ k vị trí nặng nhất ở mỗi bước, chuẩn hóa lại."""

    def edit(a, M):
        keep = np.argsort(-a, 1)[:, :k]
        b = np.zeros_like(a)
        np.put_along_axis(b, keep, np.take_along_axis(a, keep, 1), 1)
        return b / b.sum(1, keepdims=True)

    return edit


edits = {"bình thường": None, "đều": uniform, "chỉ 1 từ nặng nhất": top_k(1), "2 từ nặng nhất": top_k(2)}
acc_edit = {name: exact("add", 12, f).mean() for name, f in edits.items()}
for name, v in acc_edit.items():
    print(f"{name:>20}: đọc đúng {pct(v)} số 12 chữ số")
assert acc_edit["bình thường"] == 1 and acc_edit["đều"] == 0
assert round(N_TEST * acc_edit["chỉ 1 từ nặng nhất"]) == 80 and round(N_TEST * acc_edit["2 từ nặng nhất"]) == 295

Ba kết quả, mỗi cái dạy một điều:
- **Đều → 0%.** Mô hình thật sự *dùng* trọng số chú ý: bỏ chúng đi thì không đọc nổi một số 12 chữ số nào. Đây là phép
  thử “trọng số đều” mà Wiegreffe & Pinter đề xuất. Họ thấy ở ba bài phân loại mà Jain & Wallace dùng, trọng số đều gần như
  không làm kết quả kém đi, nên các bài ấy không dùng được để hỏi attention có giải thích hay không.
- **Chỉ 1 từ → khoảng một phần tư.** Thông tin của một chữ số thường trải trên hai trạng thái liền nhau (như “tư” và
  “tỷ” ở trên), nên “từ nặng nhất” không phải là lời giải thích đầy đủ.
- **2 từ → gần như đủ.** Phần lớn trọng số còn lại chỉ là nhiễu nhỏ.

Kết luận: trọng số chú ý cho thấy mô hình **nhìn vào đâu**, nhưng muốn biết **vì sao** nó dự đoán như vậy thì phải
can thiệp (đổi đầu vào, sửa trọng số) rồi đo lại, như ta vừa làm.

## 7 · Tích vô hướng và vì sao chia $\sqrt{d_k}$

Luong, Pham & Manning (2015) chấm điểm gọn hơn bằng tích vô hướng giữa trạng thái bộ giải mã và $h_j$, không cần
$W_a, U_a, v_a$ (họ dùng trạng thái của bước hiện tại; các mô hình tích vô hướng của khóa học dùng $s_{i-1}$ như Bahdanau). Transformer
(Vaswani và cộng sự, 2017) dùng $q \cdot k / \sqrt{d_k}$, viết cho cả câu một lần:

$$\operatorname{Attention}(Q, K, V) = AV, \qquad A = \operatorname{softmax}\!\left(\frac{QK^{\top}}{\sqrt{d_k}}\right)$$

Mỗi hàng của $Q, K, V$ là một token; softmax lấy theo từng hàng. Lý do chia: nếu các thành phần của $q$ và $k$ độc lập,
trung bình 0, phương sai 1, thì $q \cdot k = \sum_{t=1}^{d_k} q_t k_t$ là tổng của $d_k$ số hạng phương sai 1, nên có
phương sai $d_k$. Các tác giả cho rằng khi $d_k$ lớn, điểm lớn đẩy softmax vào vùng gần như one-hot, nơi gradient rất nhỏ.

Ô dưới rút $q$ và 8 khóa với $d_k = 64$ (64 là $d_k$ của Transformer gốc) bằng đúng bộ sinh số ngẫu nhiên của demo
(mulberry32 + Box–Muller, seed 1), để số in ra trùng với thẻ “Vì sao chia √dₖ” trên trang bài học.

In [ ]:
def mulberry32(seed, n):
    """n số đều trong [0, 1) từ mulberry32(seed), giống hệt hàm JavaScript của demo."""
    a = ((seed + np.arange(1, n + 1, dtype=np.uint64) * 0x6D2B79F5) & 0xFFFFFFFF).astype(np.uint32)
    t = (a ^ (a >> 15)) * (a | 1)
    t = (t + (t ^ (t >> 7)) * (t | 61)) ^ t
    return (t ^ (t >> 14)).astype(np.float64) / 2**32


def gaussian(seed, n):
    """n số N(0, 1) bằng Box–Muller, dùng cả cos lẫn sin của mỗi cặp, như demo."""
    u = mulberry32(seed, n + n % 2)
    radius, theta = np.sqrt(-2 * np.log(1 - u[0::2])), 2 * np.pi * u[1::2]
    return np.stack([radius * np.cos(theta), radius * np.sin(theta)], 1).ravel()[:n]


def softmax(z):
    a = np.exp(z - z.max(-1, keepdims=True))
    return a / a.sum(-1, keepdims=True)


dk = 64
g = gaussian(1 * 104729 + dk, dk + 8 * dk)  # cách demo đánh số seed: seed · 104729 + dₖ
q, K = g[:dk], g[dk:].reshape(8, dk)
scores = K @ q
raw, scaled = softmax(scores), softmax(scores / np.sqrt(dk))
print("q·k của 8 khóa:", " ".join(vn(v, 1) for v in scores))
print("không chia:", " ".join(pct(v, 1) for v in raw))
print("chia √64:  ", " ".join(pct(v, 1) for v in scaled))
assert abs(scores[3] - 16.2747) < 1e-3 and abs(raw.max() - 0.9974) < 1e-4 and abs(scaled.max() - 0.4821) < 1e-4

Không chia, khóa thứ tư nhận 99,7% trọng số: softmax gần như chọn cứng một khóa. Chia $\sqrt{64} = 8$, cùng các điểm ấy
cho trọng số trải ra nhiều khóa. Một lần rút chưa nói được gì; ô dưới lặp lại 500 lần cho mỗi $d_k$ từ 1 đến 1.024
(cùng cách rút của demo, seed 2026) và đo ba thứ:
- phương sai của $q \cdot k$ (lý thuyết: $d_k$);
- trọng số lớn nhất trung bình, và tỉ lệ số lần một khóa nhận hơn 99%;
- độ lớn gradient của softmax $\lVert \partial a / \partial z \rVert$ (chuẩn Frobenius của ma trận Jacobi
  $\operatorname{diag}(a) - a^{\top} a$): gradient đi ngược qua softmax nhỏ đến đâu.

In [ ]:
DKS, DRAWS, DMAX = [1, 4, 16, 64, 256, 1024], 500, 1024
G = gaussian(2026, DRAWS * 9 * DMAX).reshape(DRAWS, 9 * DMAX)  # mỗi lần rút: q (1.024 số) rồi 8 khóa × 1.024 số


def jacobian_norm(a):
    """‖diag(a) − aᵀa‖ của softmax, cho mỗi hàng của a."""
    J = a[:, :, None] * np.eye(a.shape[1]) - a[:, :, None] * a[:, None, :]
    return np.sqrt((J**2).sum((1, 2)))


stats = {}
for d in DKS:
    q_, K_ = G[:, :d], G[:, DMAX:].reshape(DRAWS, 8, DMAX)[:, :, :d]
    e_ = np.einsum("bkt,bt->bk", K_, q_)  # 500 × 8 điểm
    row = {"var": (e_**2).mean()}
    for name, z in (("raw", e_), ("scaled", e_ / np.sqrt(d))):
        a = softmax(z)
        row[name] = (a.max(1).mean(), (a.max(1) > 0.99).mean(), jacobian_norm(a).mean())
    stats[d] = row
    print(
        f"dₖ = {vn(d, 0):>5}: var(q·k) = {vn(row['var'], 1):>7} · lớn nhất {vn(row['raw'][0], 3)} → {vn(row['scaled'][0], 3)}"
        f" · >99%: {pct(row['raw'][1], 1):>6} → {pct(row['scaled'][1], 1)}"
        f" · ‖∂a/∂z‖ {vn(row['raw'][2], 3)} → {vn(row['scaled'][2], 3)}"
    )
s64 = stats[64]
assert abs(s64["var"] - 63.7691) < 1e-3 and abs(s64["raw"][0] - 0.8809) < 1e-4 and abs(s64["scaled"][0] - 0.3698) < 1e-4
assert s64["raw"][1] == 0.402 and s64["scaled"][1] == 0, "khác với demo"

Hình dưới vẽ hai cột cuối theo $d_k$ (trục ngang log): không chia (cam, ô vuông) và chia $\sqrt{d_k}$ (xanh, chấm tròn).

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 3.4))
for key, color, marker, label in (("raw", ORANGE, "s", "không chia"), ("scaled", BLUE, "o", "chia √dₖ")):
    a1.plot(DKS, [stats[d][key][0] for d in DKS], marker=marker, color=color, label=label)
    a2.plot(DKS, [stats[d][key][2] for d in DKS], marker=marker, color=color, label=label)
a1.set(
    xscale="log",
    ylim=(0, 1),
    xlabel="dₖ",
    ylabel="trọng số lớn nhất (trung bình)",
    title="Softmax nhọn dần nếu không chia",
)
a2.set(
    xscale="log", ylim=(0, 0.45), xlabel="dₖ", ylabel="‖∂a/∂z‖ (trung bình)", title="…và gradient qua softmax nhỏ dần"
)
a1.legend(frameon=False)
for ax in (a1, a2):
    ax.grid(alpha=0.3)
vn_axes(fig)
plt.show()

Không chia, trọng số lớn nhất tiến dần về 1 khi $d_k$ tăng, và gradient đi ngược qua softmax co lại: ở $d_k = 1.024$
nó chưa bằng một phần tám so với bản có chia. Chia $\sqrt{d_k}$ thì cả hai gần như không đổi theo $d_k$. Ở
$d_k = 1$ hai cách là một.

### Giả định “phương sai 1” trên mô hình thật
Lập luận trên giả định mỗi thành phần của $q$ và $k$ có phương sai 1. Trạng thái GRU thì khác: chúng đi qua $\tanh$
và các cổng, nên nhỏ hơn nhiều. Ô dưới đo trên mô hình cộng tính đã học (300 số 12 chữ số): phương sai các thành phần
của $h_j$, rồi thử chấm điểm kiểu Transformer $e_j = h_T \cdot h_j / \sqrt{64}$ trên chính các trạng thái đó, và đo
**entropy** của trọng số (bit; trọng số đều trên $S$ từ cho $\log_2 S$ bit, dồn hết vào một từ cho 0 bit).

In [ ]:
def entropy_bits(A):
    """Entropy (bit) của mỗi phân phối trọng số theo trục cuối; 0·log 0 = 0."""
    return -np.where(A > 0, A * np.log2(np.maximum(A, 1e-300)), 0).sum(-1)


Hs12, hT12 = encode(MODELS["add"], X12, M12)
var_h = Hs12[M12 > 0].var()
e_dot = np.einsum("bsh,bh->bs", Hs12, hT12) / np.sqrt(64)
uniform_bits = np.log2(M12.sum(1)).mean()
_, A12 = run("add", X12, M12, 13)
ent_add = entropy_bits(A12).mean()
print(f"phương sai thành phần của hⱼ: {vn(var_h, 3)} (giả định của √dₖ: 1)")
print(f"điểm h_T·hⱼ/8: độ lệch chuẩn {vn(e_dot[M12 > 0].std(), 2)}")
print(f"entropy: chấm điểm kiểu Transformer {vn(entropy_bits(softmax_masked(e_dot, M12)).mean(), 2)} bit")
print(f"         attention cộng tính đã học  {vn(ent_add, 2)} bit   (đều hẳn: {vn(uniform_bits, 2)} bit)")
add_att = stored("add")["seeds"][0]["att"]
assert abs(var_h - add_att["var"]) < 1e-4 and abs(ent_add - add_att["entropy"]) < 1e-4, "khác với demo"

Các thành phần của $h_j$ có phương sai khoảng 0,14 chứ không phải 1, nên $h_T \cdot h_j / 8$ cho những điểm rất nhỏ và
softmax gần như đều (3,77 bit, gần mức đều hẳn 4,08 bit). Mô hình cộng tính đã học thì tập trung hơn nhiều (1,45 bit).

Lần quét của khóa học có huấn luyện thật hai mô hình tích vô hướng trên trạng thái GRU, H = 64: chia $\sqrt{64}$
(`dot`) và không chia (`dotu`, như Luong). Ô dưới đọc kết quả từ tệp quét: entropy của attention và tỉ lệ đọc đúng ở
12 chữ số, cho cả 3 seed, cùng phương sai các thành phần của trạng thái GRU $h_j$ của chính mỗi mô hình.

In [ ]:
rows = [
    ("không attention", "none", 6000),
    ("cộng tính", "add", 6000),
    ("chia √64", "dot", 6000),
    ("chia √64, 24.000 bước", "dot", 24000),
    ("không chia", "dotu", 6000),
]
print(f"{'':>22}  {'entropy (bit)':>14}  {'phương sai hⱼ':>14}  {'đúng ở 12 chữ số':>18}")
for label, att, steps in rows:
    seeds = stored(att, steps=steps)["seeds"]
    ent = span([s["att"]["entropy"] for s in seeds], lambda v: vn(v, 2)) if "att" in seeds[0] else "—"
    var = span([s["att"]["var"] for s in seeds], lambda v: vn(v, 2)) if "att" in seeds[0] else "—"
    print(f"{label:>22}  {ent:>14}  {var:>14}  {span([s['test']['em'][-1] for s in seeds]):>18}")
print(
    f"(đều hẳn: {vn(stored('dot')['seeds'][0]['att']['uniform'], 2)} bit; phương sai hⱼ lúc khởi tạo: "
    f"{span([s['init']['var'] for s in stored('dot')['seeds']], lambda v: vn(v, 3))})"
)

Đây là một kết quả đáng nhớ:
- **Chia $\sqrt{64}$ trên trạng thái GRU:** attention gần như đều (4,05–4,06 bit, so với 4,08 bit nếu đều hẳn), mà
  trung bình đều của các $h_j$ lại là một vectơ cố định, tức gần như quay về nút thắt. Mô hình vẫn đạt 77,67–83,67% ở
  12 chữ số, gần mức không attention hơn là mức cộng tính; học lâu gấp 4 thì lên 98,33–99,33%, nhưng attention **vẫn**
  gần đều: nó học đọc mà không nhìn đúng chỗ.
- **Không chia:** đạt 100,00% ở mọi seed, với attention tập trung (0,94–1,05 bit).

Vậy $\sqrt{d_k}$ không phải là phép màu: nó là một thang đo đúng khi các thành phần có phương sai gần 1. Ở mô hình
chia $\sqrt{64}$, trạng thái GRU còn nhỏ hơn nữa (phương sai 0,07–0,08), chúng nằm trong $(-1, 1)$ và không có ma trận
nào để học lại thang đo, nên chia thêm cho 8 làm điểm quá phẳng. Trong Transformer, $q$ và $k$ là các phép chiếu đã
học $xW_q$, $xW_k$ của các vectơ vừa được chuẩn hóa (Bài 10), nên mô hình tự chỉnh được độ lớn của điểm. (Bahdanau
không gặp chuyện này: $v_a$ và $W_a, U_a$ tự học được thang đo của điểm.)

## 8 · Self-attention: một câu tự chú ý vào chính nó

Trong **tự chú ý** (self-attention), truy vấn, khóa và giá trị đều tính từ cùng một chuỗi. Bộ mã hóa ở đây thay GRU
bằng đúng một lớp như vậy (`reel/attention/s2s_sa.py`), với $X_0$ là các embedding của từ (mỗi hàng một từ):

$$Q = X_0 W_q,\quad K = X_0 W_k,\quad V = X_0 W_v,\qquad H = \tanh\!\left(X_0 + \operatorname{softmax}\!\left(\frac{QK^{\top}}{\sqrt{d_k}}\right) V\right)$$

Không có vòng lặp nào đọc từ trái sang phải: mỗi từ nối thẳng tới mọi từ, kể cả chính nó, trong một bước. Bộ giải mã
vẫn là GRU với attention cộng tính, bắt đầu từ trung bình của các hàng của $H$. Có hai phiên bản: `sa` không có gì về vị
trí, `sapos` cộng thêm một vectơ vị trí đã học $P_j$ vào từ thứ $j$ (như `wpe` của GPT-2).

In [ ]:
def encode_self(p, X, M):
    """Bộ mã hóa tự chú ý. Trả về Hs (B, S, H), trạng thái đầu của bộ giải mã s₀ và ma trận chú ý A (B, S, S)."""
    X0 = p["Ex"][X] + (p["Pe"][: X.shape[1]][None] if "Pe" in p else 0)  # chỉ sapos có vectơ vị trí
    Q, K, V = X0 @ p["Wq"], X0 @ p["Wk"], X0 @ p["Wv"]
    A = softmax_masked(np.einsum("bih,bjh->bij", Q, K) / np.sqrt(Q.shape[-1]), M[:, None, :])  # mỗi hàng cộng = 1
    Hs = np.tanh(X0 + A @ V)
    s0 = (Hs * M[:, :, None]).sum(1) / M.sum(1, keepdims=True)  # trung bình các từ: không phụ thuộc thứ tự
    return Hs, s0, A


def run_self(name, X, M, steps):
    p = MODELS[name]
    Hs, s0, A = encode_self(p, X, M)
    return decode(p, Hs, s0, M, steps, "add")[0], A


def ids(words):
    return np.array([[W2I[w] for w in words]]), np.ones((1, len(words)))


for name in ("sa", "sapos"):
    ok = [int((run_self(name, *TEST[d][:2], D_MAX + 1)[0][:, : d + 1] == TEST[d][2]).all(1).sum()) for d in TEST]
    print(f"{name:>5}: số đọc đúng trên 300 ở d = 1…12:", ok, f"· tổng {vn(sum(ok), 0)}/{vn(D_MAX * N_TEST, 0)}")
    assert ok == seed0_counts(name), "khác với demo: hãy kiểm tra lại mô hình"

Có vectơ vị trí, mô hình đọc đúng cả 3.600 số kiểm tra. Không có, nó chỉ đúng hết ở các số 1 chữ số; từ 2 chữ số trở đi
tỉ lệ đúng tụt nhanh, đến 12 chữ số chỉ còn 1 số trên 300.

### 🤔 Dự đoán trước
Đưa “hai mươi ba” (23) và “ba mươi hai” (32) vào mô hình **không** vị trí. Nó đọc ra hai số khác nhau, hay một số?

In [ ]:
out_sa, out_pos = {}, {}
for n in (23, 32):
    w = read(n)
    out_sa[n] = digits_of(run_self("sa", *ids(w), 4)[0][0])
    out_pos[n] = digits_of(run_self("sapos", *ids(w), 4)[0][0])
    print(f"“{' '.join(w)}” → không vị trí: {out_sa[n]}   có vị trí: {out_pos[n]}")
assert out_sa == {23: "23", 32: "23"} and out_pos == {23: "23", 32: "32"}, "khác với reel và demo"

Cùng một câu trả lời, 23, cho hai câu khác nhau. Lý do nằm ngay trong công thức: không có $P_j$, hàng thứ $i$ của $H$ chỉ
phụ thuộc vào *từ* thứ $i$ và *tập hợp* các từ, không phụ thuộc vào chỗ đứng của chúng. Xáo thứ tự các từ thì các hàng
của $Q, K, V$ đổi chỗ theo, ma trận $A$ đổi chỗ cả hàng lẫn cột theo cùng cách, và $H$ chỉ đổi chỗ các hàng: self-attention
**hoán vị đẳng biến** (permutation-equivariant). Sau đó trạng thái đầu là trung bình các hàng, và attention của bộ giải mã
là một tổng có trọng số qua mọi vị trí, nên đầu ra **bất biến** với mọi cách xếp. Ô dưới kiểm tra cả hai điều:
- ma trận chú ý của “ba mươi hai” đúng bằng ma trận của “hai mươi ba” với hàng và cột đổi chỗ;
- cả 6 cách xếp ba từ, và cả 5.040 cách xếp 7 từ của 2026, đều cho cùng một đầu ra.

In [ ]:
w23, w32 = read(23), read(32)
perm = [w23.index(w) for w in w32]  # vị trí trong "hai mươi ba" của từng từ trong "ba mươi hai"
A23, A32 = run_self("sa", *ids(w23), 4)[1][0], run_self("sa", *ids(w32), 4)[1][0]
print("ma trận chú ý của “hai mươi ba” (hàng: từ hỏi, cột: từ được nhìn):")
for w, row in zip(w23, A23):
    print(f"{w:>5}:", "  ".join(vn(v, 3) for v in row))
gap = np.abs(A32 - A23[perm][:, perm]).max()
print("sai khác lớn nhất giữa A(ba mươi hai) và A(hai mươi ba) đã đổi chỗ:", "0" if gap == 0 else vn(gap, 15))
assert gap < 1e-12


def all_orders(n, name="sa"):
    """Đầu ra của mô hình cho mọi cách xếp các từ đọc số n."""
    w = read(n)
    X = np.array([[W2I[w[k]] for k in order] for order in itertools.permutations(range(len(w)))])
    out, _ = run_self(name, X, np.ones(X.shape), len(str(n)) + 2)
    return len(X), {digits_of(r) for r in out}


for n in (23, 105, 2026):
    count, outs = all_orders(n)
    print(f"{' '.join(read(n))}: {vn(count, 0)} cách xếp → các đầu ra khác nhau: {sorted(outs)}")
assert all_orders(23)[1] == {"23"} and all_orders(105)[1] == {"501"} and all_orders(2026) == (5040, {"2062"})

Sai khác bằng đúng 0: phép đổi chỗ chỉ sắp xếp lại cùng những con số. Mô hình không vị trí đọc mọi cách xếp của “một
trăm linh năm” thành 501 và mọi cách xếp của 2026 thành 2062: nó chỉ thấy *túi từ*, và với mỗi túi nó đành chọn một
số. Không mô hình nào chỉ thấy túi từ có thể làm tốt hơn việc luôn chọn **số hay gặp nhất** trong số các số có cùng
túi từ. Ô dưới tính chính xác mức trần đó cho $d \le 5$, bằng cách liệt kê mọi số và xác suất của nó dưới bộ lấy mẫu.

In [ ]:
def prob(n):
    """Xác suất của số n dưới sample(), khi đã biết số chữ số của nó."""
    s = str(n)
    p_digit = lambda c: P_ZERO if c == "0" else (1 - P_ZERO) / 9
    lead = s[: len(s) - 3 * ((len(s) - 1) // 3)]  # nhóm đầu (1–3 chữ số), chữ số đầu đều trên 1–9
    p = 1 / 9 * math.prod(p_digit(c) for c in lead[1:])
    for g0 in range(len(lead), len(s), 3):
        grp = s[g0 : g0 + 3]
        p *= (1 - P_GROUP) * math.prod(p_digit(c) for c in grp) + (P_GROUP if grp == "000" else 0)
    return p


def bag_ceiling(d):
    """Tỉ lệ đúng tốt nhất của mọi bộ đọc chỉ thấy túi từ: với mỗi túi, luôn trả lời số có xác suất lớn nhất."""
    best = defaultdict(float)
    for n in range(10 ** (d - 1), 10**d):
        bag = tuple(sorted(read(n)))
        best[bag] = max(best[bag], prob(n))
    return sum(best.values())  # tổng xác suất của mọi số có d chữ số là 1


ceiling = [bag_ceiling(d) for d in range(1, 6)]
ok_sa = [int(v) for v in seed0_counts("sa")]
for d, c in enumerate(ceiling, 1):
    got = ok_sa[d - 1] / N_TEST
    print(
        f"d = {d}: trần {pct(c)} · mô hình không vị trí (seed 0) {pct(got)} · cách trần {vn(100 * (c - got), 2)} điểm"
    )
assert np.allclose(ceiling, sweep["ceiling"][:5], atol=5e-5), "khác với demo"

Mô hình không vị trí nằm ngay dưới mức trần ở mọi độ dài ta tính được (cách nhiều nhất 4,51 điểm; ở 3 chữ số: 54,00%
so với trần 54,71%): nó đã học tốt
nhất có thể *với những gì nó thấy được*, và cái nó thấy chỉ là túi từ. Hình dưới vẽ cả lần quét, 3 seed mỗi mô hình
(đường: trung vị, dải: thấp nhất–cao nhất): có vị trí (xanh, chấm tròn), không vị trí (cam, ô vuông), và mức trần của
mọi bộ đọc túi từ (xám, chữ x; tệp quét có thêm $d = 6$).

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.6))
for att, color, marker, label in (("sapos", BLUE, "o", "có vị trí"), ("sa", ORANGE, "s", "không vị trí")):
    em = 100 * em_seeds(att)
    ax.fill_between(ds, em.min(0), em.max(0), color=color, alpha=0.2)
    ax.plot(ds, np.median(em, 0), marker=marker, color=color, label=label)
ceil_all = sweep["ceiling"]
ax.plot(
    range(1, len(ceil_all) + 1), 100 * np.array(ceil_all), "x--", color=GREY, ms=8, label="trần của mọi bộ đọc túi từ"
)
ax.set(xlabel="số chữ số d", ylabel="đọc đúng cả số (%)", xticks=ds, ylim=(-2, 102))
ax.set_title("Bộ mã hóa tự chú ý, 3 seed: không có vị trí thì không biết thứ tự")
ax.legend(frameon=False)
ax.grid(axis="y", alpha=0.3)
vn_axes(fig)
plt.show()
print(f"có vị trí, mọi độ dài và mọi seed: {span(em_seeds('sapos').ravel())}")
print(f"không vị trí, 12 chữ số: {em12('sa')} · trần ở d = 6: {pct(ceil_all[5])}")

Có vectơ vị trí, cả 3 seed đọc đúng 100,00% ở mọi độ dài: một lớp tự chú ý cộng với vị trí đủ cho bài này. Không có, mọi
seed dừng ở cùng một đường, ngay dưới mức trần: đó là giới hạn của thông tin, không phải của việc học. Vì vậy mọi
Transformer đều phải thêm vị trí bằng cách nào đó: GPT-2 cộng một vectơ vị trí đã học vào mỗi token (Bài 17), các mô
hình mới hơn xoay $q$ và $k$ theo vị trí (RoPE).

### Đường tắt cho gradient
Attention còn giúp việc học theo một cách khác. Không attention, tín hiệu lỗi của chữ số đầu tiên phải đi ngược qua cả
22 bước GRU mới đến từ thứ nhất. Có attention, mỗi $h_j$ nối thẳng tới bộ giải mã, giống kết nối tắt ở
[Bài 13](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/resnet/). Lần quét đã đo $\lVert \partial L_1 / \partial x_j \rVert$
lúc khởi tạo (lỗi ở chữ số đầu, theo đầu vào của từng từ), trung bình trên 64 số 12 chữ số đọc bằng đúng 23 từ, rồi
chia cho giá trị ở từ cuối:

In [ ]:
def sci(v):
    """Số dương kiểu 3,1·10⁻⁴."""
    e = math.floor(math.log10(v))
    return f"{vn(v / 10**e, 1)}·10{str(e).translate(SUP)}" if not -1 <= e <= 2 else vn(v, 2)


g_none = [s["gradInit"][0] for s in stored("none")["seeds"]]  # log10 ‖∂L₁/∂x₁‖ / ‖∂L₁/∂x₂₃‖
g_add = [s["gradInit"][0] for s in stored("add")["seeds"]]
print(f"từ thứ nhất nhận (so với từ cuối): không attention {span([10**v for v in g_none], sci)}")
print(f"                                   có attention    {span([10**v for v in g_add], sci)}")
print(f"gấp {span([10 ** (a - n) for a, n in zip(g_add, g_none)], lambda v: vn(v, 0))} lần (theo từng seed)")

Lúc khởi tạo, không attention, từ thứ nhất chỉ nhận vài phần vạn gradient của từ cuối: tín hiệu co lại qua mỗi bước GRU,
như Bài 09 và Bài 15. Có attention, nó nhận khoảng một phần năm, gấp vài trăm lần. Mô hình không attention vẫn học được
cách giữ tín hiệu qua 22 bước bằng các cổng GRU (mục 3 cho thấy nó lên 98,33–98,67% nếu học đủ lâu), nhưng attention có
sẵn đường tắt ấy ngay từ đầu, nên học nhanh hơn.

## 9 · Thử thay đổi

Các ô dưới chạy lại cùng các hàm với tham số khác. Bạn có thể sửa giá trị rồi chạy lại.

### Đọc một số bất kỳ
Đổi các số trong danh sách. Đúng hết tập kiểm tra không có nghĩa là đúng với **mọi** số: tập kiểm tra rút theo cùng
phân phối với lúc học, còn những số “lạ” như 12 chữ số giống hệt nhau thì gần như không bao giờ được rút.

In [ ]:
for n in [2026, 15, 1000005, 100000000000, 101010101010, 999999999999]:
    X, M = ids(read(n))
    got = {name: digits_of(run(name, X, M, 16)[0][0]) for name in ("none", "add")}
    marks = {k: "đúng" if v == str(n) else "SAI" for k, v in got.items()}
    print(f"{vn(n, 0):>17}: không att. {got['none']:>14} ({marks['none']}) · có att. {got['add']:>13} ({marks['add']})")
assert digits_of(run("add", *ids(read(999999999999)), 16)[0][0]) == "99999999999"

Mô hình có attention đọc 999.999.999.999 thiếu một chữ số 9: với 12 chữ số giống hệt nhau, các trạng thái $h_j$ rất
giống nhau và nó lạc mất đang ở nhóm nào. Còn mô hình không attention lại đọc “một trăm tỷ” thành 13 chữ số.

### Độ nhọn của softmax
Nhân các điểm $e_{ij}$ của mô hình cộng tính đã học với một hệ số $k$ trước softmax ($k = 1$ là bình thường, $k = 0$ là
trọng số đều, $k \to \infty$ là chỉ giữ một từ). Đây là cùng câu hỏi với $\sqrt{d_k}$ ở mục 7: thang đo của điểm quyết
định softmax phẳng hay nhọn.

In [ ]:
def sharpen(k):
    """Sửa trọng số bằng cách nhân điểm với k: softmax(k·e) = a^k / Σ a^k."""

    def edit(a, M):
        if k == 0:
            return M / M.sum(1, keepdims=True)
        if np.isinf(k):
            return np.eye(a.shape[1])[a.argmax(1)]
        b = np.where(M > 0, np.exp(k * np.log(np.maximum(a, 1e-300))), 0)
        return b / b.sum(1, keepdims=True)

    return edit


for k in [0, 0.25, 0.5, 1, 2, 4, np.inf]:
    label = "∞" if np.isinf(k) else vn(k, 2).rstrip("0").rstrip(",")
    print(f"k = {label:>4}: đọc đúng {pct(exact('add', 12, sharpen(k)).mean()):>7} số 12 chữ số")

Quá phẳng ($k \le 0{,}5$) hay quá nhọn ($k \ge 4$) đều làm mô hình đọc kém; nó được huấn luyện với đúng $k = 1$, và
thang đo ấy là một phần của những gì nó học được.

### Tập kiểm tra khác
Rút một tập 12 chữ số mới (đổi `seed`) và so hai mô hình. Với 300 số, chênh lệch vài điểm phần trăm giữa các tập là
bình thường.

In [ ]:
for seed in (3000, 3001, 3002):
    X, M, Y = encode_set(sample(np.random.default_rng(seed), 12, N_TEST))
    acc = {name: (run(name, X, M, 13)[0] == Y).all(1).mean() for name in ("none", "add")}
    print(f"seed {seed}: không attention {pct(acc['none'])} · có attention {pct(acc['add'])}")

## 10 · Giới hạn và bước tiếp theo

- **Vẫn còn vòng lặp.** Bộ mã hóa và bộ giải mã GRU đọc và viết từng bước một, nên không tính song song theo chuỗi được.
  Transformer (Vaswani và cộng sự, 2017) bỏ hẳn RNN: mọi lớp đều là tự chú ý cộng với một mạng MLP nhỏ.
- **Một đầu, một lớp.** Bộ mã hóa tự chú ý ở mục 8 chỉ có một ma trận chú ý. Transformer chạy nhiều “đầu” song song
  (multi-head), mỗi đầu một bộ $W_q, W_k, W_v$, và xếp chồng nhiều lớp.
- **Chi phí bậc hai.** Ma trận $A$ có $S \times S$ ô: câu dài gấp đôi thì tốn gấp bốn. Với câu 23 từ thì không sao, với
  văn bản hàng trăm nghìn token thì rất đắt (Bài 25 tìm cách tránh).
- **Sinh văn bản.** GPT chỉ có phần giải mã, và *che* ma trận chú ý để mỗi token chỉ nhìn về trước. Phép che ấy cũng
  phá tính đối xứng hoán vị, nên các mô hình nhiều lớp có che vẫn suy ra được phần nào vị trí ngay cả khi không có
  vectơ vị trí (Haviv và cộng sự, 2022); bộ mã hóa không che của ta thì không.

Bài 17 ghép các mảnh này thành Transformer và GPT-2, và nhìn vào 144 bản đồ chú ý của một mô hình thật.

## Tóm tắt
- Ép cả câu vào một vectơ cố định là nút thắt: H = 64 chỉ đọc đúng 72,33–75,00% số 12 chữ số sau 6.000 bước, lỗi chủ yếu
  là đảo chỗ hai chữ số ở giữa số (964.131.195.884 → 964.131.915.884). Nhưng đó là nút thắt của việc *học*: học lâu gấp
  4, nó lên 98,33–98,67%.
- Attention: điểm $e_{ij}$ đo mức khớp giữa truy vấn $s_{i-1}$ và khóa $h_j U_a$; softmax biến điểm thành trọng số dương
  cộng bằng 1; ngữ cảnh là trung bình có trọng số của các giá trị $h_j$. Mô hình cộng tính đọc đúng 100,00%.
- Bản đồ chú ý tự học gióng hàng, nhưng nó cho thấy mô hình *nhìn* vào đâu, không phải *vì sao*: chữ số 4 nhìn vào “tỷ”
  (76,45%), dù đổi “tư” mới đổi được chữ số đó. Bỏ attention (trọng số đều) → 0%; chỉ giữ 1 từ → 26,67%.
- $\operatorname{Var}(q \cdot k) = d_k$ khi các thành phần có phương sai 1; chia $\sqrt{d_k}$ giữ softmax không quá
  nhọn (trọng số lớn nhất 0,881 → 0,370 ở $d_k = 64$). Trên trạng thái GRU (phương sai 0,07–0,08 ở chính mô hình đó)
  thì chia làm attention gần đều (4,06 bit), và mô hình chỉ đạt 77,67–83,67%.
- Tự chú ý không có vị trí là hoán vị đẳng biến: “ba mươi hai” → 23, mọi cách xếp 7 từ của 2026 → 2062, sát mức trần
  của mọi bộ đọc túi từ. Thêm vectơ vị trí: 100,00% ở mọi độ dài.

## Câu hỏi ôn tập
1. Mô hình không attention đọc sai 83 số 12 chữ số, phần lớn bằng cách đảo hai chữ số liền nhau. Điều đó nói gì về những
   gì $h_T$ còn giữ và những gì nó đánh mất?
2. Viết ba công thức của một bước attention cộng tính. Ở Bahdanau, khóa và giá trị của vị trí $j$ là gì?
3. Khi viết chữ số 4 của 964.131.195.884, trọng số lớn nhất rơi vào “tỷ” chứ không phải “tư”. Vì sao? Phép thử nào cho
   thấy chữ số đó thật sự phụ thuộc vào “tư”?
4. Nếu các thành phần của $q, k \in \mathbb{R}^{d_k}$ độc lập, trung bình 0, phương sai 1, thì $q \cdot k$ có phương
   sai bao nhiêu? Vì sao điều đó làm softmax nhọn và gradient nhỏ?
5. Trên trạng thái GRU, chia $\sqrt{64}$ làm attention gần như đều mà mô hình vẫn đọc đúng 77,67–83,67% số 12 chữ số.
   Nó đọc bằng cách nào? Vì sao mô hình không chia lại tốt hơn?
6. Vì sao bộ mã hóa tự chú ý không có vị trí trả lời giống nhau cho “hai mươi ba” và “ba mươi hai”? Nó có thể đúng tối
   đa bao nhiêu phần trăm số 3 chữ số?

<details><summary>Gợi ý đáp án</summary>

1. Ở 50 trên 83 số sai, mọi chữ số đều có mặt, chỉ hai chữ số liền nhau đổi chỗ: $h_T$ vẫn giữ *những chữ số nào*
   có trong số, nhưng thứ tự ở giữa số bị nhòe. Sau 6.000 bước, mô hình chưa học xong cách nhồi cả vị trí vào 64 số.
2. $e_{ij} = \tanh(s_{i-1}W_a + h_jU_a)\,v_a$; $\alpha_{ij} = \operatorname{softmax}_j(e_{ij})$;
   $c_i = \sum_j \alpha_{ij} h_j$. Khóa là $h_j$ (qua $U_a$), giá trị cũng là $h_j$.
3. $h_j$ là trạng thái *sau khi* GRU đọc các từ đến $j$, nên trạng thái ở “tỷ” đã chứa “tư” cùng với hàng tỷ. Đổi “tư”
   thành “ba” làm chữ số đổi thành 3, còn cấm nhìn vào “tư” thì không đổi gì: thông tin nằm ở trạng thái khác.
4. $d_k$. Điểm có độ lệch chuẩn $\sqrt{d_k}$, nên khoảng cách giữa điểm lớn nhất và các điểm khác lớn dần; softmax lấy
   mũ nên dồn gần hết trọng số vào một khóa, và Jacobi $\operatorname{diag}(a) - a^{\top}a$ tiến về 0.
5. Trung bình gần đều của mọi $h_j$ là một vectơ gần như cố định, nên nó đọc giống mô hình không attention: nhồi số vào
   các trạng thái. Không chia, điểm đủ lớn để softmax tập trung, nên nó thật sự nhìn đúng chỗ (0,94–1,05 bit).
6. Không có vectơ vị trí, mỗi trạng thái chỉ phụ thuộc vào từ của nó và tập hợp các từ; trạng thái đầu là trung bình,
   attention của bộ giải mã là tổng có trọng số. Hai câu có cùng túi từ nên cùng đầu ra. Trần ở 3 chữ số là 54,71%.
</details>

## Tìm hiểu thêm
- I. Sutskever, O. Vinyals, Q. Le (2014), [Sequence to Sequence Learning with Neural Networks](https://arxiv.org/abs/1409.3215).
- K. Cho và cộng sự (2014), [On the Properties of Neural Machine Translation: Encoder–Decoder Approaches](https://arxiv.org/abs/1409.1259):
  chất lượng giảm nhanh khi câu dài ra.
- D. Bahdanau, K. Cho, Y. Bengio (2014), [Neural Machine Translation by Jointly Learning to Align and Translate](https://arxiv.org/abs/1409.0473):
  attention cộng tính; RNNsearch-50 không kém đi kể cả với câu từ 50 từ trở lên.
- M.-T. Luong, H. Pham, C. Manning (2015), [Effective Approaches to Attention-based Neural Machine Translation](https://arxiv.org/abs/1508.04025):
  điểm tích vô hướng.
- A. Vaswani và cộng sự (2017), [Attention Is All You Need](https://arxiv.org/abs/1706.03762): $\sqrt{d_k}$ (mục 3.2.1).
- D. Britz và cộng sự (2017), [Massive Exploration of Neural Machine Translation Architectures](https://arxiv.org/abs/1703.03906):
  so sánh attention cộng tính và nhân.
- S. Jain, B. Wallace (2019), [Attention is not Explanation](https://arxiv.org/abs/1902.10186) và S. Wiegreffe,
  Y. Pinter (2019), [Attention is not not Explanation](https://arxiv.org/abs/1908.04626).
- A. Haviv và cộng sự (2022), [Transformer Language Models without Positional Encodings Still Learn Positional Information](https://arxiv.org/abs/2203.16634).
- d2l.ai, [chương Attention Mechanisms and Transformers](https://d2l.ai/chapter_attention-mechanisms-and-transformers/index.html):
  truy vấn–khóa–giá trị, attention Bahdanau, self-attention và mã hóa vị trí, có code.

**Bài tiếp theo:** [Bài 17 · Transformer & GPT](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/transformer/): xếp chồng
tự chú ý nhiều đầu, che để chỉ nhìn về trước, và đọc bên trong GPT-2.

In [ ]:
import platform

import matplotlib

print(f"Python {platform.python_version()} · numpy {np.__version__} · matplotlib {matplotlib.__version__}")